In [93]:
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/msp689/amazon-kindle-review/all_kindle_review.csv')
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [94]:
df = df[['reviewText','rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [95]:
df.shape

(12000, 2)

## Data Cleaning

In [96]:
df.isna().sum()

reviewText    0
rating        0
dtype: int64

In [97]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [98]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [99]:
df['rating'] = df['rating'].apply(lambda x :0 if x<3 else 1)

In [100]:
df['rating'].unique()

array([1, 0])

In [101]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [102]:
## Lower casing
df['reviewText'] = df['reviewText'].str.lower()

In [103]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [104]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from bs4 import BeautifulSoup ##Beautiful Soup is a library that makes it easy to scrape information from web pages. It sits atop an HTML or XML parser, providing Pythonic idioms for iterating, searching, and modifying the parse tree.               

In [105]:
## Removing special characters
df['reviewText']=df['reviewText'].apply(lambda x:re.sub('[^a-z A-z 0-9-]+', '',x))
## Remove the stopswords
df['reviewText']=df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))
## Remove url 
df['reviewText']=df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
## Remove html tags
df['reviewText']=df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())
## Remove any additional spaces
df['reviewText']=df['reviewText'].apply(lambda x: " ".join(x.split()))

In [106]:
lemmatizer=WordNetLemmatizer()

def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])
df['reviewText']=df['reviewText'].apply(lambda x:lemmatize_words(x))

## Train-Test-Split

In [107]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['reviewText'],df['rating'],test_size=0.20,random_state=42)

## BOW

In [108]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
X_train_bow=bow.fit_transform(X_train).toarray()
X_test_bow=bow.transform(X_test).toarray()

# TFIDF

In [109]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train).toarray()
X_test_tfidf=tfidf.transform(X_test).toarray()

## MODEL TRAINING

In [110]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow=GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf=GaussianNB().fit(X_train_tfidf,y_train)

In [111]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
y_pred_bow=nb_model_bow.predict(X_test_bow)
y_pred_tfidf=nb_model_bow.predict(X_test_tfidf)


In [112]:
confusion_matrix(y_test,y_pred_bow)

print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))

BOW accuracy:  0.5745833333333333


In [113]:
confusion_matrix(y_test,y_pred_tfidf)

print("TFIDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))

TFIDF accuracy:  0.57375


# ---------------------------------------------------------------------------------------------------------
# WORD2VEC IMPLEMENTAION FROM SCRATCH

In [114]:
from gensim.utils import simple_preprocess
from nltk.tokenize import sent_tokenize

In [115]:
words_train = []
for sent in X_train:
    sent_token = sent_tokenize(sent)
    for sent in sent_token:
        words_train.append(simple_preprocess(sent))

In [116]:
words_test= []
for sent in X_test:
    sent_token = sent_tokenize(sent)
    for sent in sent_token:
        words_test.append(simple_preprocess(sent))

In [117]:
words_train[0]

['looking',
 'forward',
 'book',
 'came',
 'double',
 'space',
 'every',
 'paragraph',
 'kindle',
 'edition',
 'since',
 'action',
 'move',
 'around',
 'formatting',
 'make',
 'story',
 'hard',
 'follow',
 'die',
 'hard',
 'like',
 'want',
 'botherits',
 'sad',
 'thing',
 'good',
 'book',
 'spoiled',
 'formatting',
 'fault',
 'author',
 'story',
 'good',
 'book',
 'energy',
 'read',
 'itive',
 'also',
 'emailed',
 'author']

In [118]:
words_test[0]

['really', 'great', 'read', 'wish', 'would', 'hope', 'find', 'author']

In [119]:
print(X_train.shape)
print(len(words_train))
print(X_test.shape)
print(len(words_test))

(9600,)
9600
(2400,)
2400


In [120]:
import gensim
model_train = gensim.models.Word2Vec(words_train,vector_size=100)
model_test= gensim.models.Word2Vec(words_test,vector_size=100)

In [121]:
model_train.wv.index_to_key[:20]

['book',
 'story',
 'read',
 'one',
 'character',
 'like',
 'good',
 'would',
 'really',
 'love',
 'time',
 'get',
 'author',
 'reading',
 'series',
 'well',
 'much',
 'first',
 'even',
 'short']

In [122]:
model_test.wv.index_to_key[:20]

['book',
 'story',
 'read',
 'one',
 'character',
 'like',
 'would',
 'good',
 'love',
 'really',
 'get',
 'author',
 'time',
 'well',
 'series',
 'reading',
 'first',
 'much',
 'didnt',
 'even']

In [123]:
model_train.corpus_count 

9600

In [124]:
model_test.corpus_count 

2400

In [125]:
import numpy as np
def avg_word2vec_train(doc):
    vectors = [model_train.wv[word] for word in doc if word in model_train.wv.index_to_key]

    if len(vectors) == 0:
        return np.zeros(100) 
    
    return np.mean(vectors, axis=0)

def avg_word2vec_test(doc):
    vectors = [model_test.wv[word] for word in doc if word in model_test.wv.index_to_key]

    if len(vectors) == 0:
        return np.zeros(100) 
    
    return np.mean(vectors, axis=0)

In [126]:
from tqdm import tqdm

In [127]:
X_train_word2vec=[]
for i in tqdm(range(len(X_train))):
    X_train_word2vec.append(avg_word2vec(words_train[i]))

100%|██████████| 9600/9600 [00:15<00:00, 620.50it/s]


In [128]:
X_test_word2vec=[]
for i in tqdm(range(len(X_test))):
    X_test_word2vec.append(avg_word2vec(words_test[i]))

100%|██████████| 2400/2400 [00:04<00:00, 590.71it/s]


In [129]:
X_train_word2vec = pd.DataFrame(X_train_word2vec)
X_test_word2vec = pd.DataFrame(X_test_word2vec)

In [130]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

In [131]:
rf.fit(X_train_word2vec,y_train)

RandomForestClassifier()

In [132]:
y_pred_word2vec = rf.predict(X_test_word2vec)

In [133]:
confusion_matrix(y_test,y_pred_word2vec)

print("Word2Vec: ",accuracy_score(y_test,y_pred_word2vec))

Word2Vec:  0.7675
